# Tian & Pearl Probabilities of Causation (1999)

In [2]:
import numpy as np
import pulp
import pyagrum as gum
import pyagrum.causal as csl
import pulp

from inference import ConservativePID
from symbolic import Event, P, Variable

## Setup and data generation

In [3]:
#  --- Ground Truth Model (X -> Y) ---
bn = gum.fastBN("X->Y")
bn.cpt("X")[:] = [0.5, 0.5]  # P(X=0)=0.5, P(X=1)=0.5
bn.cpt("Y")[{"X": 0}] = [0.7, 0.3]  # P(Y=1|X=0) = 0.3
bn.cpt("Y")[{"X": 1}] = [0.2, 0.8]  # P(Y=1|X=1) = 0.8

# --- Observational Data P(x,y) ---
ie = gum.LazyPropagation(bn)
ie.makeInference()
p_joint = ie.jointPosterior({"X", "Y"})

# Notation: P(x, y) where x=1 is True, y=1 is True
P_xy = p_joint[{"X": 1, "Y": 1}]  # P(x, y)
P_xy_p = p_joint[{"X": 1, "Y": 0}]  # P(x, y')
P_xp_y = p_joint[{"X": 0, "Y": 1}]  # P(x', y)
P_xp_yp = p_joint[{"X": 0, "Y": 0}]  # P(x', y')

print(
    f"Data: P(xy)={P_xy:.2f}, P(xy')={P_xy_p:.2f}, P(x'y)={P_xp_y:.2f}, P(x'y')={P_xp_yp:.2f}"
)

observational_data = {(1, 1): P_xy, (1, 0): P_xy_p, (0, 1): P_xp_y, (0, 0): P_xp_yp}

Data: P(xy)=0.40, P(xy')=0.10, P(x'y)=0.15, P(x'y')=0.35


## CPID

In [4]:
X_var = Variable("X", (0, 1))
Y_var = Variable("Y", (0, 1))

cpid = ConservativePID([X_var, Y_var], observational_data)

# Query 1: PNS = P(Y_{X=1}=1, Y_{X=0}=0)
q_pns = P(Event({Y_var @ {X_var: 1}: 1}) & Event({Y_var @ {X_var: 0}: 0}))
cpid_pns = cpid.infer(q_pns)

# Query 2: PN = P(Y_{X=0}=0 | X=1, Y=1)
q_pn = P(Event({Y_var @ {X_var: 0}: 0}), Event({X_var @ {}: 1, Y_var @ {}: 1}))
cpid_pn = cpid.infer(q_pn)

# Query 3: PS = P(Y_{X=1}=1 | X=0, Y=0)
q_ps = P(Event({Y_var @ {X_var: 1}: 1}), Event({X_var @ {}: 0, Y_var @ {}: 0}))
cpid_ps = cpid.infer(q_ps)

2026-01-27 21:21:01.881 | INFO     | inference:infer:83 - Found 1 valid causal orders compatible with the query.
2026-01-27 21:21:01.882 | INFO     | inference:infer:95 - Processing Order 1/1: ['X', 'Y']
2026-01-27 21:21:01.882 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.
2026-01-27 21:21:02.480 | INFO     | inference:infer:83 - Found 1 valid causal orders compatible with the query.
2026-01-27 21:21:02.481 | INFO     | inference:infer:95 - Processing Order 1/1: ['X', 'Y']
2026-01-27 21:21:02.481 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.
2026-01-27 21:21:02.516 | INFO     | inference:infer:83 - Found 1 valid causal orders compatible with the query.
2026-01-27 21:21:02.516 | INFO     | inference:infer:95 - Processing Order 1/1: ['X', 'Y']
2026-01-27 21:21:02.517 | INFO     | canonical:__init__:42 - Generated Basis with 8 worlds.


## Paper's LP

In [5]:
def solve_tian_pearl(objective_type):
    """
    Constructs the LP from Tian & Pearl (2000), Section 4.1.
    objective_type: 'PNS', 'PN', or 'PS'
    Returns: (min_val, max_val)
    """

    # --- Variables: The 8 Response Patterns p_ijk ---
    # i corresponds to y_x (response to X=1)
    # j corresponds to y_x' (response to X=0)
    # k corresponds to X (observed X)
    # 1=True, 0=False
    p111 = pulp.LpVariable("p111", lowBound=0)
    p110 = pulp.LpVariable("p110", lowBound=0)
    p101 = pulp.LpVariable("p101", lowBound=0)
    p100 = pulp.LpVariable("p100", lowBound=0)
    p011 = pulp.LpVariable("p011", lowBound=0)
    p010 = pulp.LpVariable("p010", lowBound=0)
    p001 = pulp.LpVariable("p001", lowBound=0)
    p000 = pulp.LpVariable("p000", lowBound=0)

    # --- Constraints (Eq 15-17 in Paper) ---

    # 1. Probability Sum = 1
    # (Implicitly handled if we constrain the partitions equal to observed sums, but good practice)
    normalisation_constraint = (
        p111 + p110 + p101 + p100 + p011 + p010 + p001 + p000 == 1
    )

    # 2. Observational Consistency (Eq 16)
    # P(x, y)   = p(y_x=1, y_x'=?, x=1) -> x=1 implies y=y_x=1
    # P(x, y)   = p111 + p101
    constraint_xy = p111 + p101 == P_xy

    # P(x, y')  = p(y_x=0, y_x'=?, x=1) -> x=1 implies y=y_x=0
    # P(x, y')  = p011 + p001
    constraint_xyp = p011 + p001 == P_xy_p

    # P(x', y)  = p(y_x=?, y_x'=1, x=0) -> x=0 implies y=y_x'=1
    # P(x', y)  = p110 + p010
    constraint_xpy = p110 + p010 == P_xp_y

    # P(x', y') = p(y_x=?, y_x'=0, x=0) -> x=0 implies y=y_x'=0
    # P(x', y') = p100 + p000
    constraint_xpyp = p100 + p000 == P_xp_yp

    # --- Objectives (Eq 18-20) ---
    # PNS = P(y_x, y'_x') = p101 + p100
    # PN  = P(y'_x' | x, y) = p101 / P(x, y)
    # PS  = P(y_x | x', y') = p100 / P(x', y')

    if objective_type == "PNS":
        objective = p101 + p100
    elif objective_type == "PN":
        # Since P(x,y) is constant, we maximize/minimize p101
        # Then divide result by P(x,y)
        objective = p101
    elif objective_type == "PS":
        # Maximize/minimize p100, divide by P(x',y')
        objective = p100

    bounds = []
    for sense in [pulp.LpMinimize, pulp.LpMaximize]:
        prob = pulp.LpProblem("TianPearlPoc", sense)

        # Add constraints
        prob += constraint_xy
        prob += constraint_xyp
        prob += constraint_xpy
        prob += constraint_xpyp
        prob += normalisation_constraint

        # Set objective
        prob += objective

        # Solve
        prob.solve(pulp.PULP_CBC_CMD(msg=False))

        val = pulp.value(prob.objective)

        # Normalize for Conditional Queries
        if objective_type == "PN":
            val /= P_xy
        elif objective_type == "PS":
            val /= P_xp_yp

        bounds.append(val)

    return bounds


pulp_pns = solve_tian_pearl("PNS")
pulp_pn = solve_tian_pearl("PN")
pulp_ps = solve_tian_pearl("PS")

In [14]:
print("\n" + "=" * 70)
print(f"{'QUERY':<5}\t|\t{'CPID (Bounds)':<20}\t|\t{'PULP (Tian & Pearl)':<20}")
print("=" * 70)
print(
    f"{'PNS':<5}\t|\t[{cpid_pns[0]:.4f}, {cpid_pns[1]:.4f}]\t|\t[{pulp_pns[0]:.4f}, {pulp_pns[1]:.4f}]"
)
print(
    f"{'PN':<5}\t|\t[{cpid_pn[0]:.4f}, {cpid_pn[1]:.4f}]\t|\t[{pulp_pn[0]:.4f}, {pulp_pn[1]:.4f}]"
)
print(
    f"{'PS':<5}\t|\t[{cpid_ps[0]:.4f}, {cpid_ps[1]:.4f}]\t|\t[{pulp_ps[0]:.4f}, {pulp_ps[1]:.4f}]"
)
print("=" * 70)

# Verification of T&P Formula Eq (21) for Observational Data:
# 0 <= PNS <= P(x,y) + P(x',y')
pns_upper_theoretical = P_xy + P_xp_yp
print(f"\nVerification:")
print(f"Theoretical Obs Upper Bound PNS (P(xy)+P(x'y')): {pns_upper_theoretical:.4f}")
print(f"Pulp PNS Upper Bound:                            {pulp_pns[1]:.4f}")
print(f"Pulp PNS Lower Bound:                            {pulp_pns[0]:.4f}")
print(f"CPID PNS Upper Bound:                            {cpid_pns[1]:.4f}")
print(f"CPID PNS Lower Bound:                            {cpid_pns[0]:.4f}")




QUERY	|	CPID (Bounds)       	|	PULP (Tian & Pearl) 
PNS  	|	[0.0000, 0.7500]	|	[0.0000, 0.7500]
PN   	|	[0.0000, 1.0000]	|	[0.0000, 1.0000]
PS   	|	[0.0000, 1.0000]	|	[0.0000, 1.0000]

Verification:
Theoretical Obs Upper Bound PNS (P(xy)+P(x'y')): 0.7500
Pulp PNS Upper Bound:                            0.7500
Pulp PNS Lower Bound:                            0.0000
CPID PNS Upper Bound:                            0.7500
CPID PNS Lower Bound:                            0.0000
